# MultiJudge — first real leaderboard on a free Colab GPU

Runs an **open-weight judge** over real multilingual data (XL-Sum) and produces a reliability leaderboard. No paid API, no proprietary data.

**Before running:** set the runtime to a GPU — `Runtime → Change runtime type → T4 GPU`.

### 1. Install dependencies
`torch` is preinstalled on Colab; the rest are free.

In [ ]:
!pip -q install transformers accelerate bitsandbytes datasets sentencepiece

### 2. Upload the package
Run the cell, choose the `babeljudge.zip` you were given, then we unzip it and put it on the path.

In [ ]:
from google.colab import files
up = files.upload()   # pick babeljudge.zip


In [ ]:
import zipfile, os, sys, glob
zname = next(n for n in up if n.endswith('.zip'))
with zipfile.ZipFile(zname) as z: z.extractall('.')
# find the directory that contains the `babeljudge` package
root = next(os.path.dirname(p) for p in glob.glob('**/babeljudge/__init__.py', recursive=True))
sys.path.insert(0, root)
import babeljudge as mj
print('babeljudge', mj.__version__, 'loaded from', root)

### 3. Confirm the GPU is visible

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

### 4. Load real multilingual data
XL-Sum human summaries. Keep it small for a first run; scale up later. (XL-Sum is CC BY-NC-SA 4.0 — fine for research; document it in the dataset card.)

In [ ]:
from babeljudge import from_xlsum, build_dataset
LANGS = ['english', 'swahili', 'hindi', 'arabic']   # mix of resource levels
sources = from_xlsum(languages=LANGS, n_per_lang=8, split='test')
items = build_dataset(sources, severities=(0.3, 0.6), seed=7)
print(f'{len(sources)} references -> {len(items)} gold-labeled items '
      f'across {len({i.language for i in items})} languages')

### 5. Build the judge
`Qwen2.5-7B-Instruct` in 4-bit fits a T4. Alternatives: `CohereForAI/aya-expanse-8b` (stronger multilingual), or `Qwen/Qwen2.5-3B-Instruct` for a tighter memory budget. Add more models to the list to compare them on the same leaderboard.

In [ ]:
from babeljudge import TransformersJudge
judges = [
    TransformersJudge('Qwen/Qwen2.5-7B-Instruct', load_in_4bit=True),
    # TransformersJudge('CohereForAI/aya-expanse-8b', load_in_4bit=True),
]

### 6. Run the benchmark
Each item is judged under both presentation orders, so position bias and order-consistency are measured. ~800 judge calls for the default slice — expect roughly 5–10 minutes on a T4.

In [ ]:
from babeljudge import evaluate, cards_to_markdown, save_results
results = evaluate(judges, items)
save_results(results, 'real_results.json', 'real_leaderboard.md')
print('saved real_results.json and real_leaderboard.md')

### 7. View the leaderboard + reliability cards

In [ ]:
from IPython.display import Markdown, display
display(Markdown(cards_to_markdown(results)))

### What to look for
- **Accuracy** that drops in lower-resource languages (swahili/yoruba) vs english — the central XL-judge finding.
- **position_bias_delta** far from 0 — the judge favors a slot.
- **verbosity_susceptibility** > 0.5 — fooled by longer-but-redundant answers.
- **reliability_score** vs raw accuracy — where a high-accuracy judge is actually unreliable.

### Next steps toward the paper
1. Scale to ~12–15 languages and 25–50 refs each; add 2–3 more open judges.
2. Add a localized-instruction run (set `instruction=` per language) for the prompt-language probe.
3. Hand-validate ~50 items to report concordance between the synthetic gold labels and human judgment.
4. Push to GitHub + a Hugging Face dataset/Space, post to arXiv, submit to the NeurIPS Evaluations & Datasets track.